# 내 업무 Agent 직접 완성하기

이 노트북에서 조회 도구 → LangChain → Graph → 수정 루프 → MCP → A2A를 이어갑니다. 코드는 셀에 작성합니다. Python 파일로 옮기는 단계는 없습니다. 해당 장의 셀까지만 실행합니다. 미완성 셀의 NotImplementedError는 구현할 부분입니다.

함수를 고쳤다면 그 정의 셀을 먼저 실행하고 아래 연결·실행 셀도 다시 실행합니다. 저장은 Ctrl+S, 셀 실행은 Shift+Enter입니다. 키는 기존 workshop/.env에서 읽으며 셀에 입력하지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, json
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "build_lab" / "materials.py").is_file():
    raise RuntimeError("workshop/notebooks에서 이 노트북을 여십시오.")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from langchain.agents import create_agent
from langchain.tools import tool
from course.policy_store import search_policy
from langgraph.graph import StateGraph, START, END
from build_lab.materials import POLICIES, Inquiry, inspect_draft, get_model, trace_messages
print("실행 위치:", root)
print("정책 주제:", list(POLICIES))

## 1A. 조회 도구

data/policies.csv와 제공 search_policy(topic)을 사용합니다. CSV 읽기·공백 처리·JSON 생성은 제공 함수가 맡습니다. lookup_policy는 이를 호출해 그대로 반환하고, docstring으로 도구의 용도를 설명합니다. build_agent에서는 tool(policy_tool)로 등록할 도구를 만듭니다.

In [ ]:
def lookup_policy(topic: str) -> str:
    """Look up the current internal policy by topic, such as 정산 or 계정."""
    return search_policy(topic)

In [ ]:
for topic in ["정산", " 계정 ", "없는업무"]:
    data = json.loads(lookup_policy(topic))
    print(data)
    assert data["found"] == (topic.strip() in POLICIES)
    assert data["topic"] == topic.strip()

## 1B. LangChain Agent

model, tools, system_prompt를 지정해 Agent를 반환합니다. 제공된 model과 policy_tool을 사용합니다. 도구 조회 시점과 정책이 없을 때의 행동을 지침에 적습니다. 다음 셀은 실제 모델 호출입니다.

In [ ]:
def build_agent(model, policy_tool):
    return create_agent(model=model, tools=[tool(policy_tool)], system_prompt=
        "사내 문의에 답하기 전에 lookup_policy로 규정을 확인하십시오. "
        "topic은 정산, 계정 같은 업무명입니다. 등록된 담당 팀과 근거 ID를 답하십시오. "
        "정책이 없으면 추가 확인을 요청하고 근거를 만들지 마십시오.")

In [ ]:
model = get_model()
agent = build_agent(model, lookup_policy)
result = agent.invoke({"messages": [{"role": "user", "content": '{"topic":"계정"}'}]}, config={"recursion_limit": 12})
for entry in trace_messages(result["messages"]):
    print(json.dumps(entry, ensure_ascii=False, indent=2))

**완료 확인:** 도구 요청·도구 결과·최종 답변에서 P-02와 IT지원팀을 찾습니다. 질문을 없는업무로 바꿔 다시 실행해 담당 팀을 지어내지 않는지 확인합니다.

## 2. 업무 Graph

공통 과정은 route_inquiry를 작성하고 제공 build_workflow를 읽습니다. 전체 노드 연결 구현은 선택 심화입니다. lookup 다음에 조건을 검사합니다. 정책이 있고 contact.strip()이 비어 있지 않으면 draft→review, 그렇지 않으면 ask로 끝냅니다. 노드는 변경할 State 필드만 반환합니다. visited에 실제 노드 이름을 순서대로 남깁니다. 웹 교재의 노드별 입출력 표를 보고 작성합니다.

In [ ]:
def route_inquiry(state):
    return "draft" if state["data"]["found"] and state.get("contact", "").strip() else "ask"

def build_workflow(lookup, generate, refine, limit=2):
    def read(state):
        return {"data": json.loads(lookup(state["topic"])), "visited": ["lookup"]}
    def route(state):
        return route_inquiry(state)
    def ask(state):
        return {"decision": "ask", "draft": "업무 주제와 회신 대상을 확인해 주세요.",
                "history": [], "visited": state["visited"] + ["ask"]}
    def draft(state):
        return {"draft": generate(state["topic"]), "visited": state["visited"] + ["draft"]}
    def review(state):
        result = refine(state["draft"], state["data"], limit)
        return {"draft": result["draft"], "history": result["history"],
                "decision": result["status"], "visited": state["visited"] + ["review"]}
    graph = StateGraph(Inquiry)
    for name, node in [("lookup", read), ("ask", ask), ("draft", draft), ("review", review)]:
        graph.add_node(name, node)
    graph.add_edge(START, "lookup")
    graph.add_conditional_edges("lookup", route, {"ask": "ask", "draft": "draft"})
    graph.add_edge("ask", END)
    graph.add_edge("draft", "review")
    graph.add_edge("review", END)
    return graph.compile()

In [ ]:
def generate_answer(topic):
    reply = agent.invoke({"messages": [{"role": "user", "content": json.dumps({"topic": topic}, ensure_ascii=False)}]}, config={"recursion_limit": 12})
    return reply["messages"][-1].content

# 수정 루프를 배우기 전에는 실제 초안을 한 번 검토합니다.
def inspect_once(draft, data, limit):
    feedback = inspect_draft(draft, data)
    return {"status": "held" if feedback else "passed", "draft": draft,
            "history": [{"attempt": 0, "draft": draft, "feedback": feedback}]}


model_calls = []
def generate(topic):
    model_calls.append(topic)
    return generate_answer(topic)

graph = build_workflow(lookup_policy, generate, inspect_once)


### 2A. 실행 입력과 성공 기준

아래 셀의 topic·contact를 바꾸고 실행합니다. 계정+정상 주소는 lookup→draft→review, 빈 문자열·공백·없는업무는 lookup→ask입니다. 추가 확인 경로는 모델 호출 0회여야 합니다. 정상 경로의 held는 history의 검토 이유를 확인합니다.

In [ ]:
topic = "계정"
contact = "   "
model_calls.clear()
result = graph.invoke({"topic": topic, "contact": contact})
print("방문:", result["visited"])
print("판정:", result["decision"])
print("모델 호출:", len(model_calls))
print("답변:", result["draft"])

## 3. 제공 수정 Loop 관찰

공통 과정에서는 아래 연결 함수를 그대로 사용하고 guided.py를 읽습니다. 직접 재구현하려는 경우에만 다음 계약을 구현합니다. 성공→정체→예산 순서로 검사합니다. status는 passed, stalled, held입니다. history에는 attempt, draft, feedback을 남깁니다. 최초 검토는 attempt=0입니다. limit은 0~5 정수이며 bool은 거부합니다. 수정에는 이번 초안의 피드백을 전달합니다.

In [ ]:
def refine_answer(draft, data, revise, limit=2):
    if type(limit) is not int or not 0 <= limit <= 5:
        raise ValueError("수정 상한은 0~5 정수입니다.")
    history = []
    for attempt in range(limit + 1):
        feedback = inspect_draft(draft, data)
        history.append({"attempt": attempt, "draft": draft, "feedback": feedback})
        if not feedback:
            return {"status": "passed", "draft": draft, "history": history}
        if len(history) > 1 and history[-2]["draft"] == draft:
            return {"status": "stalled", "draft": draft, "history": history}
        if attempt == limit:
            return {"status": "held", "draft": draft, "history": history}
        draft = revise(draft, feedback)

In [ ]:
feedback_inputs = []
data = json.loads(lookup_policy("계정"))
def revise(draft, feedback):
    feedback_inputs.append({"draft": draft, "feedback": feedback})
    return model.invoke("규정에 따라 초안을 수정하십시오. 담당 팀과 근거 ID를 포함하십시오.\n" +
                        json.dumps({"policy": data, "draft": draft, "feedback": feedback}, ensure_ascii=False)).content

repaired = refine_answer("확인 완료", data, revise, 2)
print(json.dumps(repaired, ensure_ascii=False, indent=2))
print("실제로 전달한 수정 입력:", feedback_inputs)
assert feedback_inputs, "이 초안은 기준이 부족하므로 수정 호출이 필요합니다."
assert repaired["history"][0]["draft"] == "확인 완료"

## 연결하고 설명합니다

앞에서 만든 그래프에 제공 수정 루프를 연결합니다. 결과가 통과하지 않으면 기준을 낮추지 말고 history의 실패와 수정 입력을 읽습니다.

In [ ]:
def refine(draft, data, limit):
    return refine_answer(draft, data, revise, limit)
app = build_workflow(lookup_policy, generate, refine, limit=2)
final = app.invoke({"topic": "계정", "contact": "user@example.test"})
print(json.dumps(final, ensure_ascii=False, indent=2))

## 4. MCP · 같은 셀의 함수를 HTTP 도구로 공개합니다

build_mcp_server를 구현합니다. MCPServer를 만들고 policy_tool을 등록해 반환합니다. 아래 연결 셀은 서버를 시작하고 요청한 뒤 종료합니다. 별도 터미널이나 Python 파일 복사는 필요하지 않습니다.

In [ ]:
from mcp.server.mcpserver import MCPServer
from course.notebook_server import serve_app
from course.mcp_lab import query

def build_mcp_server(policy_tool):
    server = MCPServer("업무 규정")
    server.tool()(policy_tool)
    return server

In [ ]:
topic = "계정"
server = build_mcp_server(lookup_policy)
async with serve_app(server.streamable_http_app(stateless_http=True, json_response=True)) as url:
    wire = await query(url + "/mcp", topic)
print("도구 목록:", wire["tools"])
data = json.loads(wire["result"]["content"][0]["text"])
print(data)

**완료 확인:** 목록에 lookup_policy, data.policy에 P-02·IT지원팀이 있습니다. topic을 없는업무로 바꿔 found=False를 확인합니다. 노트북에는 이미 이벤트 루프가 있으므로 asyncio.run 대신 await를 사용합니다.

## 5. A2A · 원격 검토 수용 조건을 작성합니다

accept_review를 구현합니다. submitted/working은 pending, completed 외 종료는 held입니다. completed일 때 현재 요청 ID·양의 정수 버전·artifact 버전·passed is True가 모두 맞아야 accepted입니다. bool 버전과 빈 ID도 거절합니다.

In [ ]:
def accept_review(state, artifact, request_id, version):
    from course.a2a_lab import accept_result
    return accept_result(state, artifact, request_id, version)

In [ ]:
artifact = {"request_id": "case-1", "version": 1, "passed": True}
for state, request_id, version in [("working", "case-1", 1), ("completed", "case-1", 1), ("completed", "case-1", 2), ("completed", "other", 1)]:
    print(accept_review(state, artifact, request_id, version))
# 순서대로 pending, accepted, held, held

## 6. 통합 · 실제 MCP 조회와 원격 검토를 연결합니다

위 다섯 구현 셀을 완성한 뒤 실행합니다. 아래 입력을 바꾸면 MCP로 읽는 정책과 Graph 입력이 함께 바뀝니다. API 호출은 실제로 진행됩니다.

In [ ]:
import uuid
from course.a2a_lab import create_app, delegate

topic = "계정"
contact = "user@example.test"
server = build_mcp_server(lookup_policy)
async with serve_app(server.streamable_http_app(stateless_http=True, json_response=True)) as url:
    wire = await query(url + "/mcp", topic)
snapshot = json.loads(wire["result"]["content"][0]["text"])

snapshot_topics = {topic.strip(), snapshot["topic"]}

def snapshot_lookup(topic: str) -> str:
    """이번 문의에서 MCP로 조회한 규정을 반환합니다."""
    return json.dumps(snapshot if topic.strip() in snapshot_topics else {"found": False, "topic": topic, "policy": None}, ensure_ascii=False)

connected_agent = build_agent(model, snapshot_lookup)
def connected_generate(topic):
    result = connected_agent.invoke({"messages": [{"role": "user", "content": topic}]})
    return result["messages"][-1].content

def connected_refine(draft, data, limit):
    def revise_current(draft, feedback):
        return model.invoke("규정에 맞게 초안을 수정하십시오.\n" + json.dumps({"policy": data, "draft": draft, "feedback": feedback}, ensure_ascii=False)).content
    return refine_answer(draft, data, revise_current, limit)

workflow = build_workflow(snapshot_lookup, connected_generate, connected_refine)
result = workflow.invoke({"topic": topic, "contact": contact})
print("Graph 결과:", result)
if result["decision"] == "passed":
    payload = {"topic": topic, "draft": result["draft"], "request_id": str(uuid.uuid4()), "version": 1}
    async with serve_app(lambda port: create_app(port, model=model), factory=True) as url:
        review = await delegate(url, payload)
    decision = accept_review(review["state"], review["artifact"], payload["request_id"], payload["version"])
    print("원격 검토:", review)
    print("수용 판단:", decision)
else:
    print("원격 검토 전 종료:", result["decision"])

**완료 확인:** 계정+정상 주소에서 실제 조회·생성·검토를 확인합니다. 추가 확인 경로는 원격 검토 전에 끝납니다. 응답이 completed여도 요청·버전·통과 조건이 맞아야 accepted입니다. 별칭 변경 과제는 lookup_policy 셀을 수정하고 6번 셀을 재실행합니다.

Harness 설계 활동은 제공된 HARNESS_WORKSHEET.md의 F1/D1 사례를 읽고 아래 셀에 작성합니다.

## Harness 설계 메모

- 시작 조건:
- 다음 작업 선택:
- 종료·인계 조건:
- 역할별 입력과 산출물:
- 중복 알림·검토 누락·잘못된 PASS 처리: